# TASK 2.6: Thám mã Vigenère (Ciphertext-Only Attack)

### Nguyên lý hoạt động (Cơ sở toán học & Thống kê)

Để phá mã Vigenère, ta cần hóa giải sức mạnh của nó là "làm phẳng phân phối tần suất". Bản chất cốt lõi là đưa bài toán Vigenère phức tạp về thành nhiều bài toán Caesar đơn giản. Quá trình này gồm 3 bước chính:

In [10]:
#import và khai báo hằng số tần suất chuẩn
import string
from collections import Counter

# Tần suất chữ cái tiếng Anh chuẩn 
ENGLISH_FREQ = {
    'A': 0.08167, 'B': 0.01492, 'C': 0.02782, 'D': 0.04253, 'E': 0.12702,
    'F': 0.02228, 'G': 0.02015, 'H': 0.06094, 'I': 0.06966, 'J': 0.00015,
    'K': 0.00772, 'L': 0.04025, 'M': 0.02406, 'N': 0.06749, 'O': 0.07507,
    'P': 0.01929, 'Q': 0.00095, 'R': 0.05987, 'S': 0.06327, 'T': 0.09056,
    'U': 0.02758, 'V': 0.00978, 'W': 0.02360, 'X': 0.00150, 'Y': 0.01974,
    'Z': 0.00074
}

**1. Tìm từng ký tự của khóa bằng Kiểm định Chi-bình phương ($\chi^2$)**
- Sau khi có độ dài $L$, ta chia bản mã thành $L$ nhóm. Lúc này, mỗi nhóm bị mã hóa bởi cùng 1 ký tự duy nhất (tương đương mật mã Caesar).
- Xét từng nhóm, ta thử dịch chuyển ngược (brute-force) 26 khả năng khóa từ 'A' đến 'Z'. Ở mỗi phép thử, ta dùng công thức **Chi-Square** để đo lường độ lệch giữa tần suất chữ cái vừa giải mã với bảng tần suất tiếng Anh chuẩn.
- Phép dịch chuyển nào cho ra điểm $\chi^2$ thấp nhất (nghĩa là giống cấu trúc tiếng Anh nhất) sẽ là ký tự chính xác của khóa.

In [11]:
# 1. Hàm tính Index of Coincidence (IoC) cho một chuỗi
def calculate_ioc(text):
    n = len(text)
    if n <= 1:
        return 0
    counts = Counter(text)
    ioc = sum(count * (count - 1) for count in counts.values()) / (n * (n - 1))
    return ioc

# 2. Hàm tính điểm Chi-Square để so sánh với tiếng Anh
def get_chi_square(text):
    n = len(text)
    counts = Counter(text)
    chi_square = 0
    for char in string.ascii_uppercase:
        observed = counts.get(char, 0)
        expected = ENGLISH_FREQ[char] * n
        if expected > 0:
            chi_square += ((observed - expected) ** 2) / expected
    return chi_square

**2. Ước lượng độ dài khóa bằng Chỉ số trùng hợp (Index of Coincidence - IoC)**
- Mật mã Vigenère với độ dài khóa $L$ thực chất là sự đan xen của $L$ mật mã Caesar độc lập.
- Trong tiếng Anh, do sự xuất hiện không đồng đều của các chữ cái (E, T, A xuất hiện nhiều nhất), Chỉ số trùng hợp (IoC) luôn xấp xỉ **0.065**. Trong khi đó, một chuỗi ký tự ngẫu nhiên hoàn toàn chỉ có IoC khoảng **0.038**.
- Thuật toán sẽ giả sử khóa có độ dài $L$ chạy từ 1 đến 20. Với mỗi $L$, chia bản mã thành $L$ nhóm (cột) và tính IoC trung bình. Giá trị $L$ nào làm cho mức IoC trung bình "vọt" lên gần mức 0.065 nhất thì đó chính là độ dài khóa.

In [12]:
# 3. Hàm ước lượng độ dài khóa tốt nhất
def find_key_length(text, max_len=20):
    best_len = 1
    best_ioc = 0
    print("--- Phân tích IoC để tìm độ dài khóa ---")
    
    for length in range(1, max_len + 1):
        # Tách chuỗi thành `length` cột
        columns = [text[i::length] for i in range(length)]
        
        # Tính IoC trung bình của các cột
        avg_ioc = sum(calculate_ioc(col) for col in columns) / length
        
        if avg_ioc > best_ioc:
            best_ioc = avg_ioc
            best_len = length
            
    print(f">> Độ dài khóa tiềm năng nhất L = {best_len} (IoC trung bình = {best_ioc:.4f})")
    return best_len



# 4. Hàm tìm chính xác chuỗi Khóa
def find_key(text, key_length):
    key = ""
    columns = [text[i::key_length] for i in range(key_length)]
    
    for col in columns:
        best_shift = 0
        min_chi_square = float('inf')
        
        # Thử 26 khả năng Caesar cho cột này
        for shift in range(26):
            decrypted_col = ""
            for char in col:
                # Dịch ngược lại (giải mã)
                decrypted_char = chr((ord(char) - ord('A') - shift + 26) % 26 + ord('A'))
                decrypted_col += decrypted_char
                
            chi_sq = get_chi_square(decrypted_col)
            
            # Chọn bước dịch có độ lệch thấp nhất (giống tiếng Anh nhất)
            if chi_sq < min_chi_square:
                min_chi_square = chi_sq
                best_shift = shift
                
        key += chr(best_shift + ord('A'))
    return key


**3. Khôi phục bản rõ (Decryption)**
- Ghép $L$ ký tự vừa tìm được thành từ khóa hoàn chỉnh.
- Sử dụng khóa này để giải mã toàn bộ ciphertext ban đầu bằng công thức giải mã Vigenère chuẩn: $P_i = (C_i - K_{i \pmod L} + 26) \pmod{26}$, có kết hợp giữ nguyên khoảng trắng và dấu câu.

In [13]:
# 5. Hàm giải mã Vigenere giữ nguyên định dạng khoảng trắng
def vigenere_decrypt(ciphertext, key):
    plaintext = ""
    key_idx = 0
    key = key.upper()
    
    for char in ciphertext:
        if char.isalpha():
            base = ord('A') if char.isupper() else ord('a')
            c = ord(char) - base
            k = ord(key[key_idx % len(key)]) - ord('A')
            
            plaintext += chr((c - k + 26) % 26 + base)
            key_idx += 1
        else:
            plaintext += char
    return plaintext

In [14]:
# main
if __name__ == "__main__":
    ciphertext = str(input("Nhập ciphertext:"))

    print("BẢN MÃ GỐC:\n", ciphertext[:100], "...\n")

    # Bước 1: Chuẩn hóa dữ liệu (Chỉ lấy chữ cái, chuyển in hoa)
    normalized_text = "".join(filter(str.isalpha, ciphertext)).upper()

    # Bước 2: Tìm độ dài khóa
    k_len = find_key_length(normalized_text)

    # Bước 3: Tìm Khóa
    found_key = find_key(normalized_text, k_len)
    print(f">> KHÓA TÌM ĐƯỢC: {found_key}\n")

    # Bước 4: Giải mã bản rõ 
    final_plaintext = vigenere_decrypt(ciphertext, found_key)
    print("BẢN RÕ ĐÃ GIẢI MÃ:\n")
    print(final_plaintext)

BẢN MÃ GỐC:
 1 ...

--- Phân tích IoC để tìm độ dài khóa ---
>> Độ dài khóa tiềm năng nhất L = 1 (IoC trung bình = 0.0000)
>> KHÓA TÌM ĐƯỢC: A

BẢN RÕ ĐÃ GIẢI MÃ:

1
